# 02: AHDE (Adaptive Hybrid Decision Engine) Routing Model
**AI Shopping OS — Memory-Augmented Multi-Agent AI Shopping Assistant**

This notebook trains an **XGBoost Multi-Output Classifier** to predict the optimal execution graph (activation vector for 11 downstream agents) and model tier (`fast`, `standard`, `deep`) based on query complexity, twin confidence, and memory presence.


In [ ]:
# Step 1: Install Dependencies
!pip install -q xgboost lightgbm scikit-learn numpy pandas skl2onnx onnxruntime


In [ ]:
# Step 2: Generate Synthetic AHDE Training Traces
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)
N_SAMPLES = 2000

# Input Features:
# 0: Intent ID (0..7)
# 1: Query Character Length
# 2: Token Complexity (1=simple, 2=moderate, 3=complex)
# 3: Twin Confidence Score (0..100)
# 4: Memory Count (0..50)
# 5: Budget Filter Specified (0 or 1)

intent_ids = np.random.randint(0, 8, N_SAMPLES)
query_lens = np.random.randint(10, 120, N_SAMPLES)
complexities = np.random.choice([1, 2, 3], N_SAMPLES, p=[0.4, 0.4, 0.2])
twin_confs = np.random.randint(20, 100, N_SAMPLES)
mem_counts = np.random.randint(0, 50, N_SAMPLES)
has_budgets = np.random.choice([0, 1], N_SAMPLES)

X = pd.DataFrame({
    'intent_id': intent_ids,
    'query_len': query_lens,
    'complexity': complexities,
    'twin_conf': twin_confs,
    'mem_count': mem_counts,
    'has_budget': has_budgets
})

# 11 Target Agent Activations:
# [Search, Memory, Twin, MARE_Rec, Review, Price, Budget, Twin_Evolve, Explain, Cart, Notification]
AGENTS = ["Search", "Memory", "Twin", "MARE_Rec", "Review", "Price", "Budget", "Twin_Evolve", "Explain", "Cart", "Notification"]

Y_list = []
for idx, row in X.iterrows():
    intent = row['intent_id']
    comp = row['complexity']
    
    # Defaults
    activations = [0] * 11
    
    if intent == 0: # recommendation
        activations = [1, 1 if row['mem_count']>0 else 0, 1 if row['twin_conf']>40 else 0, 1, 1 if comp>=2 else 0, 1, 1 if row['has_budget'] else 0, 1, 1, 0, 0]
    elif intent == 1: # comparison
        activations = [1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0]
    elif intent == 2: # price_tracking
        activations = [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]
    elif intent == 3: # search
        activations = [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
    elif intent == 4: # budget
        activations = [0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0]
    elif intent == 5: # cart
        activations = [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0]
    elif intent == 6: # greeting
        activations = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    else: # general
        activations = [1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0]
        
    Y_list.append(activations)

Y = pd.DataFrame(Y_list, columns=AGENTS)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(f"Generated {N_SAMPLES} execution traces for 11 multi-label agents.")


In [ ]:
# Step 3: Train XGBoost Multi-Output Classifier
import xgboost as xgb
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss, accuracy_score, f1_score

base_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
multi_model = MultiOutputClassifier(base_model)

print("Training AHDE Multi-Output Routing Model...")
multi_model.fit(X_train, Y_train)

# Predictions
Y_pred = multi_model.predict(X_test)

# Metrics
subset_acc = accuracy_score(Y_test, Y_pred)
h_loss = hamming_loss(Y_test, Y_pred)
macro_f1 = f1_score(Y_test, Y_pred, average='macro')

print(f"Subset Accuracy (Exact Graph Match): {subset_acc*100:.2f}%")
print(f"Hamming Loss: {h_loss:.4f}")
print(f"Macro F1-Score: {macro_f1:.4f}")


In [ ]:
# Step 4: Model Tier Decision Classifier (Fast / Standard / Deep)
# Tier logic: 1=fast, 2=standard, 3=deep

def predict_model_tier(complexity, num_active_agents):
    if complexity == 3 or num_active_agents >= 7:
        return "deep"
    elif complexity == 2 or num_active_agents >= 4:
        return "standard"
    return "fast"

sample_query = X_test.iloc[0:1]
active_pred = multi_model.predict(sample_query)[0]
active_agents = [AGENTS[i] for i in range(len(AGENTS)) if active_pred[i] == 1]
tier = predict_model_tier(sample_query['complexity'].values[0], len(active_agents))

print("Sample Query Test:")
print("Features:", sample_query.to_dict(orient='records')[0])
print("Activated Agents:", active_agents)
print("Predicted Model Tier:", tier)


In [ ]:
# Step 5: Export AHDE Model with Joblib
import joblib

joblib.dump(multi_model, "ahde_routing_xgboost.pkl")
print("Exported model to ahde_routing_xgboost.pkl")
